In [19]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import os
import ee

# ---------------- Auth ----------------
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/explore/nobackup/people/spotter5/ee/trevors-playground-338818-64c929992646.json"
service_account = "stefano-ee@trevors-playground-338818.iam.gserviceaccount.com"
credentials = ee.ServiceAccountCredentials(
    service_account,
    "/explore/nobackup/people/spotter5/ee/trevors-playground-338818-64c929992646.json",
)
ee.Initialize(credentials)
ee.Initialize(project="trevors-playground-338818")

# ---------------- Config ----------------
YEAR = 2024
SEED = 42
NUM_TRAIN = 10000                  # random training points inside AOI
K_MIN, K_MAX = 5, 30              # inclusive

TARGET_CRS = "EPSG:3413"
TARGET_SCALE = 1000               # meters (1 km)

BUCKET = "smp-ee-files"
GCS_PREFIX = "kmeans"             # everything goes under gs://smp-ee-files/kmeans/...

# ---------------- AOI (FeatureCollection) ----------------
aoi = ee.FeatureCollection("users/spotter/tundra_and_boreal_smooth")
regionGeom = aoi.geometry()  # no 'geometry' variable anywhere

# ---------------- EC tower sites ----------------
sites = (
    ee.FeatureCollection("projects/top-operand-328213/assets/sites")
    .filter(ee.Filter.eq("flux_method", "EC"))
    .filterBounds(regionGeom)
)

# ---------------- Embeddings ----------------
EMB_ID = "GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL"
start = ee.Date.fromYMD(YEAR, 1, 1)
end = start.advance(1, "year")

# Bounds-filtered mosaic (no clip to keep masks simple)
emb_native = (
    ee.ImageCollection(EMB_ID)
    .filterDate(start, end)
    .filterBounds(regionGeom)
    .mosaic()
    .toFloat()
)

bandNames = emb_native.bandNames()
nativeProj = emb_native.select("A00").projection()
nativeScale = nativeProj.nominalScale()

# Reprojected view for point sampling at 3413 / 1 km
proj_3413_1km = ee.Projection(TARGET_CRS).atScale(TARGET_SCALE)
emb_3413_1km = emb_native.resample("bilinear").reproject(proj_3413_1km)

# ---------------- Training sample: random points inside AOI ----------------
# (Don’t print training.getInfo() to avoid memory hits.)
pts = ee.FeatureCollection.randomPoints(region=regionGeom, points=NUM_TRAIN, seed=SEED)

training = emb_native.sampleRegions(
    collection=pts,
    scale=nativeScale,
    projection=nativeProj,
    geometries=False,
    tileScale=2,
)

# ---------------- Per-K: train, export image + site CSV to GCS ----------------
def run_k(k: int):
    # Train k-means on 64-D embedding bands
    clusterer = ee.Clusterer.wekaKMeans(
        nClusters=k, seed=SEED, maxIterations=200
    ).train(features=training, inputProperties=bandNames)

    # (A) Classify the FULL AOI (raster)
    cluster_img = (
        emb_native
        .clip(regionGeom)            # limit output spatially
        .cluster(clusterer)
        .rename("cluster")
        .toInt16()
    )

    # Export raster to GCS at 1 km / EPSG:3413
    img_desc = f"embeddings_{YEAR}_k{k}_aoi"
    img_prefix = f"{GCS_PREFIX}/embeddings_{YEAR}_k{k}_aoi"
    img_task = ee.batch.Export.image.toCloudStorage(
        image=cluster_img,
        region=regionGeom,
        description=img_desc,
        fileNamePrefix=img_prefix,   # e.g., kmeans/embeddings_2024_k12_aoi
        scale=TARGET_SCALE,
        crs=TARGET_CRS,
        maxPixels=1e13,
        bucket=BUCKET,
    )
    # img_task.start()
    print(f"[GCS image] k={k} → gs://{BUCKET}/{img_prefix}.tif")

    # (B) Sample EC tower sites on the 1 km grid & assign clusters (no big raster op)
    sitePix = emb_3413_1km.sampleRegions(
        collection=sites,
        properties=["site_reference"],
        scale=TARGET_SCALE,
        projection=proj_3413_1km,
        geometries=False,
        tileScale=2,
    )

    labeledSites = sitePix.cluster(clusterer).map(lambda f: f.set({"year": YEAR, "k": k}))
    out = labeledSites.select(["site_reference", "cluster", "year", "k"])

    # Export CSV to GCS
    csv_desc = f"sites_embeddings_{YEAR}_k{k}"
    csv_prefix = f"{GCS_PREFIX}/sites_embeddings_{YEAR}_k{k}"  # → .../kmeans/sites_embeddings_2024_k12.csv
    csv_task = ee.batch.Export.table.toCloudStorage(
        collection=out,
        description=csv_desc,
        bucket=BUCKET,
        fileNamePrefix=csv_prefix,
        fileFormat="CSV",
    )
    csv_task.start()
    print(f"[GCS CSV]   k={k} → gs://{BUCKET}/{csv_prefix}.csv")

# Launch for K = 5..30
for k in range(K_MIN, K_MAX + 1):
    run_k(k)

print("All GCS exports started. Monitor in the Code Editor Tasks panel or via ee.batch.Task.list().")


[GCS image] k=5 → gs://smp-ee-files/kmeans/embeddings_2024_k5_aoi.tif
[GCS CSV]   k=5 → gs://smp-ee-files/kmeans/sites_embeddings_2024_k5.csv
[GCS image] k=6 → gs://smp-ee-files/kmeans/embeddings_2024_k6_aoi.tif
[GCS CSV]   k=6 → gs://smp-ee-files/kmeans/sites_embeddings_2024_k6.csv
[GCS image] k=7 → gs://smp-ee-files/kmeans/embeddings_2024_k7_aoi.tif
[GCS CSV]   k=7 → gs://smp-ee-files/kmeans/sites_embeddings_2024_k7.csv
[GCS image] k=8 → gs://smp-ee-files/kmeans/embeddings_2024_k8_aoi.tif
[GCS CSV]   k=8 → gs://smp-ee-files/kmeans/sites_embeddings_2024_k8.csv
[GCS image] k=9 → gs://smp-ee-files/kmeans/embeddings_2024_k9_aoi.tif
[GCS CSV]   k=9 → gs://smp-ee-files/kmeans/sites_embeddings_2024_k9.csv
[GCS image] k=10 → gs://smp-ee-files/kmeans/embeddings_2024_k10_aoi.tif
[GCS CSV]   k=10 → gs://smp-ee-files/kmeans/sites_embeddings_2024_k10.csv
[GCS image] k=11 → gs://smp-ee-files/kmeans/embeddings_2024_k11_aoi.tif
[GCS CSV]   k=11 → gs://smp-ee-files/kmeans/sites_embeddings_2024_k11.cs

Now for each K i want to determine the frequency of clusters per site_reference, for example with 100 clusters perhaps some classes only have 1 observations. 

In [20]:
import os, re, glob
import pandas as pd
import matplotlib.pyplot as plt

# ----------- CONFIG -----------
BASE_DIR = "/explore/nobackup/people/spotter5/anna_v/v2/kmeans"
OUT_DIR  = os.path.join(BASE_DIR, "summary")
os.makedirs(OUT_DIR, exist_ok=True)

FILENAME_RE = re.compile(r"_k(\d+)", re.IGNORECASE)  # pulls K from "..._k12(.csv)"

# ----------- LOAD ALL CSVs -----------
rows = []
for path in glob.glob(os.path.join(BASE_DIR, "*.csv")):
    try:
        df = pd.read_csv(path)
    except Exception as e:
        print(f"Skip {os.path.basename(path)} (read error): {e}")
        continue

    # Parse K from filename if not present
    k_match = FILENAME_RE.search(os.path.basename(path))
    k_from_name = int(k_match.group(1)) if k_match else None

    if "k" not in df.columns:
        if k_from_name is None:
            print(f"Skip {os.path.basename(path)} (can't infer k)")
            continue
        df["k"] = k_from_name

    # Keep the essential columns; tolerate column order/casing
    cols = {c.lower(): c for c in df.columns}
    needed = ["site_reference", "cluster", "k"]
    if not all(x in cols for x in needed):
        print(f"Skip {os.path.basename(path)} (missing one of {needed})")
        continue

    df = df[[cols["site_reference"], cols["cluster"], cols["k"]]].copy()
    df.columns = ["site_reference", "cluster", "k"]

    # Clean types
    df["k"] = pd.to_numeric(df["k"], errors="coerce").astype("Int64")
    df["cluster"] = pd.to_numeric(df["cluster"], errors="coerce").astype("Int64")
    df = df.dropna(subset=["k", "cluster", "site_reference"])

    rows.append(df)

if not rows:
    raise SystemExit("No valid CSVs found. Check BASE_DIR or file naming.")

data = pd.concat(rows, ignore_index=True)

# ----------- CLUSTER SIZE PER K -----------
# How many sites fell into each cluster label for each K
cluster_sizes = (
    data.groupby(["k", "cluster"])["site_reference"]
    .nunique()
    .rename("n_sites")
    .reset_index()
    .sort_values(["k", "n_sites", "cluster"], ascending=[True, False, True])
)

# Save the long table
cluster_sizes.to_csv(os.path.join(OUT_DIR, "cluster_sizes_by_k.csv"), index=False)
print(f"Saved: {os.path.join(OUT_DIR, 'cluster_sizes_by_k.csv')}")

# ----------- SUMMARY PER K -----------
def summarize_k(g):
    # g is rows for a single K
    sizes = g["n_sites"].astype(int)
    return pd.Series({
        "clusters_reported": sizes.shape[0],
        "sites_total": int(sizes.sum()),
        "min_size": sizes.min(),
        "q25_size": sizes.quantile(0.25),
        "median_size": sizes.median(),
        "mean_size": sizes.mean(),
        "q75_size": sizes.quantile(0.75),
        "max_size": sizes.max(),
        "num_singletons": int((sizes == 1).sum()),
        "num_le5": int((sizes <= 5).sum()),
    })

k_summary = cluster_sizes.groupby("k").apply(summarize_k).reset_index()
k_summary.to_csv(os.path.join(OUT_DIR, "k_summary.csv"), index=False)
print(f"Saved: {os.path.join(OUT_DIR, 'k_summary.csv')}")

# # ----------- PLOTS: sorted cluster sizes for each K -----------
# # Creates one PNG per K showing how cluster sizes distribute (big → small)
# for k_val, g in cluster_sizes.groupby("k"):
#     sizes_sorted = g.sort_values("n_sites", ascending=False)["n_sites"].tolist()
#     plt.figure(figsize=(8, 4.5))
#     plt.plot(range(1, len(sizes_sorted) + 1), sizes_sorted, marker="o")
#     plt.title(f"Cluster sizes (sorted) — K={k_val}")
#     plt.xlabel("Cluster rank (by size)")
#     plt.ylabel("# Sites")
#     plt.tight_layout()
#     png_path = os.path.join(OUT_DIR, f"cluster_sizes_k{k_val}.png")
#     plt.savefig(png_path, dpi=150)
#     plt.close()
#     print(f"Saved: {png_path}")

# ----------- PLOTS: sorted cluster sizes for each K (with labels) -----------
# ----------- PLOTS: sorted cluster sizes for each K (labels slightly higher) -----------
# ----------- PLOTS: force exactly K points (include zero-count clusters) -----------
for k_val in sorted(cluster_sizes["k"].unique()):
    # take rows for this K and reindex to all cluster ids 0..K-1 (fill missing with 0)
    g = cluster_sizes[cluster_sizes["k"] == k_val].set_index("cluster")
    g_full = (
        g.reindex(range(k_val), fill_value=0)        # ensure K rows, zeros for missing clusters
         .rename_axis("cluster")
         .reset_index()[["cluster", "n_sites"]]
    )
    # sort by size (desc) for rank plot
    g_sorted = g_full.sort_values("n_sites", ascending=False).reset_index(drop=True)
    sizes_sorted = g_sorted["n_sites"].tolist()

    plt.figure(figsize=(8, 4.5))
    x = range(1, len(sizes_sorted) + 1)
    plt.plot(x, sizes_sorted, marker="o", linestyle="-")

    # labels just above each point
    for i, val in enumerate(sizes_sorted, start=1):
        plt.text(i, val + 2, str(val), fontsize=8, ha="center", va="bottom", color="dimgray")

    plt.title(f"Cluster sizes (sorted) — K={k_val}")
    plt.xlabel("Cluster rank (by size)")
    plt.ylabel("# Sites")
    plt.ylim(bottom=0)  # start y-axis at 0 so zeros are visible on the axis
    plt.grid(alpha=0.3, linestyle="--", linewidth=0.5)
    plt.tight_layout()

    png_path = os.path.join(OUT_DIR, f"cluster_sizes_k{k_val}.png")
    plt.savefig(png_path, dpi=150)
    plt.close()
    print(f"Saved: {png_path}")
# ----------- OPTIONAL: quick views in console -----------
print("\n=== k_summary (head) ===")
print(k_summary.head(10).to_string(index=False))

print("\n=== Example: clusters with ≤ 5 sites for K=30 ===")
example = cluster_sizes[(cluster_sizes["k"] == 30) & (cluster_sizes["n_sites"] <= 5)]
print(example.to_string(index=False))


Saved: /explore/nobackup/people/spotter5/anna_v/v2/kmeans/summary/cluster_sizes_by_k.csv
Saved: /explore/nobackup/people/spotter5/anna_v/v2/kmeans/summary/k_summary.csv


/explore/nobackup/people/spotter5/temp_dir/ipykernel_2490575/2976200891.py:84: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  k_summary = cluster_sizes.groupby("k").apply(summarize_k).reset_index()


Saved: /explore/nobackup/people/spotter5/anna_v/v2/kmeans/summary/cluster_sizes_k5.png
Saved: /explore/nobackup/people/spotter5/anna_v/v2/kmeans/summary/cluster_sizes_k6.png
Saved: /explore/nobackup/people/spotter5/anna_v/v2/kmeans/summary/cluster_sizes_k7.png
Saved: /explore/nobackup/people/spotter5/anna_v/v2/kmeans/summary/cluster_sizes_k8.png
Saved: /explore/nobackup/people/spotter5/anna_v/v2/kmeans/summary/cluster_sizes_k9.png
Saved: /explore/nobackup/people/spotter5/anna_v/v2/kmeans/summary/cluster_sizes_k10.png
Saved: /explore/nobackup/people/spotter5/anna_v/v2/kmeans/summary/cluster_sizes_k11.png
Saved: /explore/nobackup/people/spotter5/anna_v/v2/kmeans/summary/cluster_sizes_k12.png
Saved: /explore/nobackup/people/spotter5/anna_v/v2/kmeans/summary/cluster_sizes_k13.png
Saved: /explore/nobackup/people/spotter5/anna_v/v2/kmeans/summary/cluster_sizes_k14.png
Saved: /explore/nobackup/people/spotter5/anna_v/v2/kmeans/summary/cluster_sizes_k15.png
Saved: /explore/nobackup/people/spott

In [9]:
# Step 1: identify the cluster with largest area per K
largest_clusters = (
    cluster_sizes
    .sort_values(["k", "n_sites"], ascending=[True, False])
    .groupby("k")
    .first()
    .reset_index()
    .rename(columns={"cluster": "largest_cluster", "n_sites": "largest_cluster_sites"})
)

# Step 2: flag whether that largest cluster has any tower representation
largest_clusters["largest_cluster_has_tower"] = largest_clusters["largest_cluster_sites"] > 0

# Step 3: add summary columns for clarity
largest_clusters["largest_cluster_empty"] = ~largest_clusters["largest_cluster_has_tower"]
largest_clusters["largest_cluster_empty_pct"] = (
    100 * largest_clusters["largest_cluster_empty"].astype(int)
)

print(largest_clusters[["k", "largest_cluster", "largest_cluster_sites", "largest_cluster_has_tower"]])

     k  largest_cluster  largest_cluster_sites  largest_cluster_has_tower
0    5                3                    120                       True
1    6                5                    103                       True
2    7                5                    103                       True
3    8                5                     94                       True
4    9                5                     73                       True
5   10                5                     61                       True
6   16                5                     63                       True
7   17                5                     49                       True
8   18                8                     56                       True
9   19                8                     52                       True
10  20                8                     52                       True
11  21                8                     52                       True
12  23                8               

In [21]:
import os
import re
import pandas as pd

# --- CONFIG ---
YEAR = 2024
K = 12  # <-- change this to any k you want to inspect
# If your files are local:
BASE_DIR = "/explore/nobackup/people/spotter5/anna_v/v2/kmeans"
# If you’re pulling directly from GCS (download first or use gcsfuse/gcsfs).
# For local files exported by your pipeline, filenames look like:
#   sites_embeddings_2024_k12.csv
FILENAME = f"sites_embeddings_{YEAR}_k{K}.csv"
PATH = os.path.join(BASE_DIR, FILENAME)

# --- LOAD ---
df = pd.read_csv(PATH)

# Tolerate column case/order; keep essentials
cols = {c.lower(): c for c in df.columns}
needed = ["site_reference", "cluster", "k"]
missing = [x for x in needed if x not in cols]
if missing:
    raise ValueError(f"Missing required columns in {PATH}: {missing}")

df = df[[cols["site_reference"], cols["cluster"], cols["k"]]].copy()
df.columns = ["site_reference", "cluster", "k"]

# Clean types and de-dup per site if necessary
df["cluster"] = pd.to_numeric(df["cluster"], errors="coerce").astype("Int64")
df["k"] = pd.to_numeric(df["k"], errors="coerce").astype("Int64")
df = df.dropna(subset=["site_reference", "cluster"])
df = df.drop_duplicates(subset=["site_reference"])  # one row per site

# --- GROUP: cluster -> sites & counts ---
cluster_lists = (
    df.groupby("cluster")["site_reference"]
      .apply(lambda s: sorted(s.unique().tolist()))
      .rename("site_references")
      .reset_index()
)

cluster_counts = (
    df.groupby("cluster")["site_reference"]
      .nunique()
      .rename("n_sites")
      .reset_index()
)

summary = (
    cluster_counts
      .merge(cluster_lists, on="cluster", how="left")
      .sort_values(["n_sites", "cluster"], ascending=[True, True])
      .reset_index(drop=True)
)

# --- PRINT nicely ---
print(f"\nK = {K} — clusters sorted by # of towers (smallest → largest)")
for _, row in summary.iterrows():
    cl = int(row["cluster"])
    n  = int(row["n_sites"])
    sites = row["site_references"]
    print(f"  • Cluster {cl:>2}  ({n} site{'s' if n!=1 else ''}): {', '.join(sites) if n>0 else '(none)'}")

# Optional: save to a CSV for record
# OUT_DIR = os.path.join(BASE_DIR, "summary")
# os.makedirs(OUT_DIR, exist_ok=True)
# out_path = os.path.join(OUT_DIR, f"sites_by_cluster_k{K}.csv")
# summary.to_csv(out_path, index=False)
# print(f"\nSaved detailed table → {out_path}")



K = 12 — clusters sorted by # of towers (smallest → largest)
  • Cluster  1  (2 sites): Elgeeii forest station_RU-Ege_tower, Southern Khentei Taiga_MN-Skt_tower
  • Cluster  9  (3 sites): Fish Island_FI-NWT_tower, Gunnarsholt_IS-Gun_tower, Umiujaq_tower
  • Cluster  5  (4 sites): Wolf_creek_Buckbrush_CA-WCBB_tower, Wolf_creek_SparseShrub_CA-WCPLT_tower, Wolf_creek_forest_CA-WCF_tower, Wolf_creek_upper_forest_CA-WCUF_tower
  • Cluster  7  (4 sites): Hustai grassland_MN-Hst_tower, Kherlenbayan Ulaan_MN-Kbu_tower, Nalaikh grassland_MN-Nkh_tower, Udleg practice forest_MN-Udg_tower
  • Cluster  2  (6 sites): Alberta - Western Peatland - Poor Fen (Sphagnum moss)_CA-WP2_tower, ZOTTO Bog_RU-Zo1_tower, ZOTTO Forest_RU-Zo2_tower, Zotino; Central Siberia_RU-Zfw 1_tower, Zotino; Central Siberia_RU-Zfw 2_tower, Zotino_RU-Zot_tower
  • Cluster  6  (8 sites): Adventdalen_SJ-Adv_tower, Bayelva, Spitsbergen_SJ-Blv_tower, Lake Hazen, Ellesmere Island_CA-LHazen1-semidesert_tower, Lake Hazen, Ellesmere I

In [22]:
df

,site_reference,cluster,k
0,Churchill Fen Site 2_CA-CF2_tower,10,12
1,"Stordalen, Sweden_Stordalen_tower",3,12
2,Wolf_creek_upper_forest_CA-WCUF_tower,5,12
3,Cascaden Ridge Fire Scar_US-Fcr_tower,4,12
4,Mukhrino_Mukhrino_RHC_tower,8,12
...,...,...,...
152,Hyytiala_FI-Hyy_tower,0,12
153,Scotty Creek Landscape_CA-SCC_tower,8,12
154,Smith Creek_CA-SMC_tower,8,12
155,Siikaneva2_FI-Si2_tower,0,12
